# Online Shoppers — Musteri Segmentasyonu

Bu notebook, ziyaretcileri davranissal ozelliklere gore gruplandirmak icin
K-Means kumeleme yontemini uygulamaktadir.
Uretilen `Segment` ozelligi, Classification notebook'unda feature olarak
kullanilarak model performansı artırılmaya çalışılacaktır.

**Akis:**
1. Veri hazirlama ve ozellik muhendisligi
2. Optimal K secimi (Elbow + Silhouette)
3. K-Means model egitimi
4. PCA ile 2B gorsellestirme
5. Segment profilleri ve yorumlama
6. Segment bazinda revenue analizi

## 1. Kutuphaneler ve Veri Yukleme

In [ ]:
import warnings

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

df = pd.read_csv("cleaned.csv", index_col=0)
print(f"Veri boyutu: {df.shape}")
df.head()

## 2. Ozellik Muhendisligi

In [ ]:
# Kategorik degiskenleri sayisala donustur
df["VisitorType_enc"] = df["VisitorType"].map(
    {"Returning_Visitor": 2, "New_Visitor": 1, "Other": 0}
)
df["Weekend_enc"] = df["Weekend"].astype(int)

month_map = {
    "Feb": 2, "Mar": 3, "May": 5, "June": 6,
    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12,
}
df["Month_enc"] = df["Month"].map(month_map)

SEG_FEATURES = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues",
    "SpecialDay", "VisitorType_enc", "Weekend_enc", "Month_enc",
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[SEG_FEATURES])

print(f"Kumeleme icin ozellik sayisi : {len(SEG_FEATURES)}")
print(f"Olceklendirilmis matris      : {X_scaled.shape}")

## 3. Optimal K Secimi

In [ ]:
inertias   = []
sil_scores = []
K_RANGE    = range(2, 11)

for k in K_RANGE:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(
        silhouette_score(X_scaled, labels, sample_size=3000, random_state=42)
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_RANGE, inertias, "o-", color="#1565c0", linewidth=2.5, markersize=8)
axes[0].axvline(x=3, color="red", linestyle="--", alpha=0.7, label="K=3")
axes[0].set_xlabel("Kume Sayisi (K)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_title("Elbow Yontemi")
axes[0].legend()

axes[1].plot(K_RANGE, sil_scores, "s-", color="#2e7d32", linewidth=2.5, markersize=8)
best_k = list(K_RANGE)[int(np.argmax(sil_scores))]
axes[1].scatter(best_k, max(sil_scores), s=200, zorder=5, color="red",
                label=f"En yuksek: K={best_k} ({max(sil_scores):.4f})")
axes[1].axvline(x=3, color="red", linestyle="--", alpha=0.7)
axes[1].set_xlabel("Kume Sayisi (K)")
axes[1].set_ylabel("Silhouette Skoru")
axes[1].set_title("Silhouette Analizi")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Silhouette'e gore optimal K = {best_k}")

## 4. K-Means Model Egitimi (K=3)

In [ ]:
K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["Segment"] = kmeans.fit_predict(X_scaled)

print("Segment dagilimi:")
for seg, cnt in df["Segment"].value_counts().sort_index().items():
    print(f"  Segment {seg}: {cnt:,} ziyaretci ({cnt / len(df) * 100:.1f}%)")

final_sil = silhouette_score(X_scaled, df["Segment"], sample_size=5000, random_state=42)
print(f"\nFinal Silhouette Skoru: {final_sil:.4f}")

## 5. PCA ile 2B Gorsellestirme

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_

print(f"PC1 aciklanan varyans : {explained[0]:.2%}")
print(f"PC2 aciklanan varyans : {explained[1]:.2%}")
print(f"Toplam               : {sum(explained):.2%}")

COLORS = {0: "#1565c0", 1: "#c62828", 2: "#2e7d32"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for seg in range(K):
    mask = df["Segment"] == seg
    axes[0].scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=COLORS[seg], label=f"Segment {seg} (n={mask.sum():,})",
        alpha=0.35, s=10,
    )
centers_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1],
                c="black", s=200, marker="X", zorder=5, label="Merkezler")
axes[0].set_xlabel(f"PC1 ({explained[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({explained[1]:.1%})")
axes[0].set_title("PCA — Segmentler")
axes[0].legend(fontsize=9)

rev_colors = df["Revenue"].astype(int).map({0: "#b0bec5", 1: "#e53935"})
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=rev_colors, alpha=0.3, s=10)
axes[1].set_xlabel(f"PC1 ({explained[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({explained[1]:.1%})")
axes[1].set_title("PCA — Revenue")
axes[1].legend(handles=[
    mpatches.Patch(color="#e53935", label="Revenue = True"),
    mpatches.Patch(color="#b0bec5", label="Revenue = False"),
], fontsize=9)

plt.tight_layout()
plt.show()

## 6. Segment Profilleri

In [ ]:
profile = df.groupby("Segment")[SEG_FEATURES].mean()

norm = MinMaxScaler()
profile_norm = pd.DataFrame(
    norm.fit_transform(profile),
    index=profile.index,
    columns=profile.columns,
)

plt.figure(figsize=(14, 4))
sns.heatmap(
    profile_norm.T,
    annot=profile.T.round(2),
    fmt=".2f",
    cmap="RdYlGn",
    linewidths=0.4,
    xticklabels=[f"Segment {i}" for i in range(K)],
    cbar_kws={"label": "Normalize Deger (0-1)"},
)
plt.title("Segment Profil Isi Haritasi (Normalize)")
plt.ylabel("Ozellik")
plt.tight_layout()
plt.show()

## 7. Segment Bazinda Revenue Analizi

In [ ]:
rev_rate = df.groupby("Segment")["Revenue"].apply(
    lambda x: x.astype(int).mean() * 100
)
seg_size = df["Segment"].value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Revenue orani
bars = axes[0].bar(
    [f"Seg {i}" for i in rev_rate.index],
    rev_rate.values,
    color=[COLORS[i] for i in rev_rate.index],
    edgecolor="white",
)
for bar, val in zip(bars, rev_rate.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{val:.1f}%", ha="center", fontweight="bold", fontsize=10,
    )
axes[0].set_title("Revenue Orani (%)")
axes[0].set_ylabel("Satin Alan Ziyaretci %")

# PageValues dagilimi
df.boxplot(
    column="PageValues", by="Segment", ax=axes[1],
    patch_artist=True,
    boxprops=dict(facecolor="#90caf9"),
    medianprops=dict(color="red", linewidth=2),
)
axes[1].set_title("PageValues Dagilimi")
axes[1].set_xlabel("Segment")
plt.sca(axes[1])
plt.title("")

# Pasta grafik
axes[2].pie(
    seg_size,
    labels=[f"Segment {i}
(n={v:,})" for i, v in seg_size.items()],
    autopct="%1.1f%%",
    colors=[COLORS[i] for i in seg_size.index],
    startangle=90,
    textprops={"fontsize": 10},
)
axes[2].set_title("Segment Buyuklugu")

plt.suptitle("")
plt.tight_layout()
plt.show()

## 8. Ziyaretci Tipi ve Ay Bazinda Segment Dagilimi

In [ ]:
month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

vt_seg = pd.crosstab(df["VisitorType"], df["Segment"], normalize="index") * 100
vt_seg.plot(
    kind="bar", ax=axes[0],
    color=[COLORS[i] for i in range(K)],
    edgecolor="white", rot=15,
)
axes[0].set_title("Ziyaretci Tipi x Segment (%)")
axes[0].set_xlabel("Ziyaretci Tipi")
axes[0].set_ylabel("%")
axes[0].legend(title="Segment", labels=[f"Seg {i}" for i in range(K)])

month_seg = pd.crosstab(df["Month"], df["Segment"])
month_seg = month_seg.reindex([m for m in month_order if m in month_seg.index])
(month_seg.div(month_seg.sum(axis=1), axis=0) * 100).plot(
    kind="bar", ax=axes[1], stacked=True,
    color=[COLORS[i] for i in range(K)],
    edgecolor="white", linewidth=0.5,
)
axes[1].set_title("Ay x Segment Dagilimi (Stacked %)")
axes[1].set_xlabel("Ay")
axes[1].set_ylabel("%")
axes[1].legend(title="Segment", labels=[f"Seg {i}" for i in range(K)])
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 9. Segment Yorumlari

**Segment 0 — Pasif Grup**
Dusuk sayfa degeri, yuksek bounce/exit rate ve az urun sayfasi ziyareti.
Siteye girer, kisa surede ayrilir; satin alma niyeti dusuktur.
Yeniden hedefleme (retargeting) kampanyalari ve dikkat cekici giris teklifleri
bu segment icin onerilmektedir.

**Segment 1 — Aktif Alicilar**
Yuksek PageValues, uzun urun sayfasi suresi, dusuk exit rate.
Urunleri inceleyen ve satin alma kararini neredeyse vermis ziyaretcilerdir.
Kisisellestirme, sepet hatirlatma ve sadakat programlari uygulanabilir.

**Segment 2 — Arastirmaci Ziyaretciler**
Yuksek informational ve administrative sayfa ziyareti, orta duzey bounce rate.
Bilgi toplar, karsilastirir; karar surecindedir.
Karsilastirma rehberleri, site içi feedbackler, remarketing ve sinirli sureli indirim
kuponlari bu segmentte etkin olabilir.


In [ ]:
df.to_csv("segmented_shoppers.csv", index=False)
print("Segmentli veri 'segmented_shoppers.csv' olarak kaydedildi.")
print(f"  Toplam satir : {len(df):,}")
print(f"  Sutunlar     : {list(df.columns)}")